# S6 — Plotly 互動 + 完整 Capstone 案例

> **時間**：2 小時（講授 60 + 實作 50 + 成果發表 10）  
> **資料**：`orders_raw.csv`（從頭走一遍完整 DE/DA 流程）  
> **學完能幹嘛**：獨立完成 raw CSV → 清理 → 分析 → 互動式儀表板的**全流程**

---

## 本節為什麼是 Capstone

前 5 個 session 我們把知識拆成碎片學：
- S1 NumPy 向量化
- S2 Pandas 清理
- S3 Pandas 合併與聚合
- S4 時序與 EDA
- S5 視覺化

**但真實工作不會這樣分章節**。你會拿到一份 raw CSV，要**一路做到老闆能點的儀表板**。本節就是把前 5 章串成一個完整專案。

**新增一個工具：Plotly**
- S5 的 matplotlib/seaborn → **靜態圖片**（交出去是 .png）
- Plotly → **互動網頁**（能 hover、縮放、篩選，能存成 .html 寄給老闆）

**本節只教 Plotly Express 6 個常用函式**（做減法）：
`px.line / px.bar / px.scatter / px.box / px.histogram / px.pie`


---
## Part A — Plotly Express 速成（30 min）


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = 'plotly_white'  # 乾淨白底

df = pd.read_csv(
    '../datasets/ecommerce/orders_enriched.csv',
    parse_dates=['order_date'],
)
print('資料形狀:', df.shape)

In [ ]:
# 1) 折線圖：月度營收
df['month'] = df['order_date'].dt.to_period('M').astype(str)
monthly = df.groupby('month', as_index=False)['amount'].sum()

fig = px.line(monthly, x='month', y='amount', markers=True,
              title='Monthly Revenue Trend')
fig.update_layout(height=400)
fig.show()

In [ ]:
# 2) 長條圖：地區營收（支援 hover、自動排序）
region_rev = df.groupby('region', as_index=False)['amount'].sum().sort_values('amount', ascending=False)
fig = px.bar(region_rev, x='region', y='amount', text='amount',
             color='region', title='Revenue by Region')
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(height=400, showlegend=False)
fig.show()

In [ ]:
# 3) 散佈圖：單價 vs 金額（hover 可顯示商品名）
fig = px.scatter(df, x='unit_price', y='amount',
                 color='category', hover_data=['product_name', 'customer_name'],
                 title='Unit Price vs Order Amount')
fig.update_layout(height=450)
fig.show()

In [ ]:
# 4) 箱型圖：品類分布
fig = px.box(df, x='category', y='amount', color='category',
             title='Order Amount by Category')
fig.update_layout(height=400, showlegend=False)
fig.show()

In [ ]:
# 5) 直方圖：金額分布
fig = px.histogram(df, x='amount', nbins=30, title='Amount Distribution')
fig.update_layout(height=400)
fig.show()

In [ ]:
# 6) 圓餅圖：VIP 等級佔比
vip_rev = df.groupby('vip_level', as_index=False)['amount'].sum()
fig = px.pie(vip_rev, names='vip_level', values='amount',
             title='VIP Level Share', hole=0.4)  # hole=0.4 變成 donut
fig.update_layout(height=400)
fig.show()

---
## Part B — Capstone：從 raw 到 dashboard 完整走一遍（50 min）

**情境**：你是公司新進 DA，主管給你一份髒的訂單 CSV，下週一要交一份互動式儀表板。  
**要做的事**：重新走 S2 → S3 → S4 → S6，中間不看之前的 notebook，能獨立寫完。


### Step 1 — 載入與清理（S2）


In [ ]:
def load_and_clean_orders(path):
    """封裝 S2 的清理邏輯，方便未來重用。"""
    df = pd.read_csv(path)
    # 欄名標準化
    df.columns = df.columns.str.strip().str.lower()
    # 金額轉數字
    df['amount'] = (
        df['amount'].astype(str)
        .str.replace('$','', regex=False)
        .str.replace(',','', regex=False)
        .astype(float)
    )
    # 日期轉 datetime
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
    # 缺值處理
    df = df.dropna(subset=['order_date'])
    df['qty'] = df['qty'].fillna(df['qty'].median())
    # 去重
    df = df.drop_duplicates()
    return df

orders = load_and_clean_orders('../datasets/ecommerce/orders_raw.csv')
print(f'清理完成：{orders.shape[0]} 筆訂單')
orders.head(3)

### Step 2 — 合併主檔（S3）


In [ ]:
customers = pd.read_csv('../datasets/ecommerce/customers.csv')
products  = pd.read_csv('../datasets/ecommerce/products.csv')

enriched = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print(f'合併後形狀: {enriched.shape}')
print(f'欄位數: {len(enriched.columns)}')

### Step 3 — 計算關鍵指標（S3 + S4）


In [ ]:
enriched['month'] = enriched['order_date'].dt.to_period('M').astype(str)

# 關鍵 KPIs
kpis = {
    '總營收':     enriched['amount'].sum(),
    '總訂單數':   len(enriched),
    '活躍顧客數': enriched['customer_id'].nunique(),
    '客單價':     enriched['amount'].sum() / len(enriched),
}
for k, v in kpis.items():
    print(f'{k}: {v:>12,.0f}')

### Step 4 — 互動式儀表板（4 張圖，使用 subplot）


> ### 💡 知識補給站 — 儀表板設計心法：KPI → Trend → Detail
> 
> 好的儀表板有**層次結構**，不是把所有圖塞在一起：
> 
> | 層次 | 放什麼 | 目的 |
> |---|---|---|
> | **最上面** | KPI 卡片（3~5 個數字） | 一眼抓住重點（總營收、訂單數、客單價） |
> | **中間** | 趨勢圖（折線） | 讓老闆知道「是在變好還是變差」 |
> | **下面** | 明細 / 排名（長條、表格） | 哪個地區、品類、商品表現最好 |
> 
> 這叫 **"KPI → Trend → Detail" 三層架構**，也是 Tableau / Power BI 的預設佈局邏輯。
> 
> 我們的 Step 3 算了 KPI、Step 4 就是趨勢 + 明細 — 你已經在做對的事了。
> 
> **注意**：Dashboards ≠ Reports — 儀表板是**持續監控**用的，報告是**解釋過去決策**用的。
> 
> *延伸關鍵字：dashboard design, KPI hierarchy, Tableau, Power BI, information architecture*

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 先算四份資料
monthly     = enriched.groupby('month', as_index=False)['amount'].sum()
top_prod    = (enriched.groupby('product_name', as_index=False)['amount']
               .sum().sort_values('amount', ascending=False).head(10))
region_rev  = enriched.groupby('region', as_index=False)['amount'].sum()
cat_rev     = enriched.groupby('category', as_index=False)['amount'].sum()

# 2x2 subplot
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Monthly Revenue Trend',
                    'Top 10 Products',
                    'Revenue by Region',
                    'Category Share'),
    specs=[[{'type':'xy'}, {'type':'xy'}],
           [{'type':'xy'}, {'type':'domain'}]],
)

fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['amount'],
                         mode='lines+markers', name='Monthly'), row=1, col=1)
fig.add_trace(go.Bar(x=top_prod['product_name'], y=top_prod['amount'],
                     name='Top Products'), row=1, col=2)
fig.add_trace(go.Bar(x=region_rev['region'], y=region_rev['amount'],
                     name='Region'), row=2, col=1)
fig.add_trace(go.Pie(labels=cat_rev['category'], values=cat_rev['amount'],
                     name='Category', hole=0.4), row=2, col=2)

fig.update_layout(
    title_text='<b>E-Commerce Sales Dashboard — 2025</b>',
    height=750, showlegend=False,
)
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.show()

### Step 5 — 存檔：把儀表板變成可以寄出去的 HTML


In [ ]:
fig.write_html('../datasets/ecommerce/dashboard.html')
print('✅ 儀表板已存成 dashboard.html，可以直接寄給老闆。')

---
## 🏆 成果發表（10 min）

每位學員輪流用 30 秒講：
1. 今天做完的儀表板，**最關鍵的發現**是什麼？
2. 如果再給你 1 小時，你會**新增哪張圖**？
3. 這份流程在你未來工作上的哪個場景最有用？


---
## 課後挑戰（自學作業）

### 🎯 Capstone 延伸
基於本節的儀表板，加入以下任一功能：
1. **時間篩選器**：用 `plotly.graph_objects` 的 `updatemenus` 讓使用者選擇不同月份
2. **顧客分群**：用 S3 挑戰題的 RFM 結果畫一張 3D 散佈圖 (`px.scatter_3d`)
3. **自動化報表**：把 `load_and_clean_orders` + 儀表板生成打包成一個 Python script，未來只要換 CSV 就能跑出新報表


---
## 小結與速查表

| 想畫什麼 | Plotly Express |
|---|---|
| 折線 | `px.line(df, x, y, markers=True)` |
| 長條 | `px.bar(df, x, y, color, text)` |
| 散佈 | `px.scatter(df, x, y, color, hover_data)` |
| 箱型 | `px.box(df, x, y, color)` |
| 直方 | `px.histogram(df, x, nbins)` |
| 圓餅 | `px.pie(df, names, values, hole=0.4)` |
| 多圖 | `make_subplots + add_trace` |
| 存檔 | `fig.write_html(path)` |

---

## 🎓 12 小時課程總結

恭喜完成全部 6 個 session！你現在會的能力：

| Session | 能力 | 對應職能 |
|---|---|---|
| S1 | 向量化運算 | 基礎 Python 資料處理 |
| S2 | 讀檔 + 清理 | **DE** 的第一哩路 |
| S3 | 多表 Join + 聚合 | **DE/DA** 核心能力 |
| S4 | 時序 + EDA | **DA** 核心能力 |
| S5 | 靜態視覺化 | 報告交付 |
| S6 | 互動儀表板 | 成品交付 |

**下一步的自學建議**：
1. 學 **SQL**（本課程的 groupby/merge 思維 100% 通用）
2. 學 **scikit-learn**（把今天的分析往 ML 延伸）
3. 學 **Streamlit** 或 **Dash**（把 notebook 變成真正的 Web App）

**記住一件事：DA/DE 的價值不在技術，在於你能回答老闆『所以呢？』這個問題。**


> ### 💡 知識補給站 — 從分析到行動：ML 與 A/B Test 的下一步
> 
> 本課程你學會了「**描述過去**」（Descriptive Analytics）— 過去賣了多少、哪個品類最好、哪個月最旺。
> 
> 但真正有商業價值的是**下一步**：
> 
> - **預測未來（Predictive）**：用今天算的特徵（月營收、客單價、RFM）餵給 **scikit-learn**，預測「這個客戶下個月會不會回購？」→ 這就是 **Feature Engineering**，你在 S3 做的 groupby + agg 就是在算特徵
> - **驗證假設（Prescriptive）**：你發現某地區營收特別高，但這是因為行銷活動還是自然需求？→ 設計 **A/B Test**，用統計檢定（t-test / chi-square）驗證差異是否顯著
> 
> 記住：**DA 的終極價值不是做圖，是回答「所以我們該怎麼做？」**
> 
> *延伸關鍵字：feature engineering, scikit-learn, A/B testing, t-test, descriptive → predictive → prescriptive analytics*